# Βήμα 7 – Ρύθμιση του δικτύου στα 8 χαρακτηριστικά (Σύστημα Α)
Στο notebook 09 το δίκτυο με τα 8 αριθμητικά χαρακτηριστικά βγήκε λίγο χειρότερο από το Random Forest.
Εκεί όμως χρησιμοποιήσαμε το δίκτυο που είχε σχεδιαστεί για 67 εισόδους. Εδώ ψάχνουμε **τι δίκτυο ταιριάζει σε 8 εισόδους**.

1. Έλεγχος: βγαίνουν τα ίδια 8 χαρακτηριστικά αν οι συσχετίσεις υπολογιστούν χωρίς τις διπλότυπες γραμμές;
2. Στόχος: το Random Forest στα ίδια 8 χαρακτηριστικά (validation και test).
3. Δοκιμάζουμε 9 δίκτυα: 3 μεγέθη × 3 τιμές dropout, με περισσότερες εποχές.
4. Διαλέγουμε **στο validation** και ελέγχουμε με άλλους σπόρους τυχαιότητας αν η διαφορά είναι πραγματική.

Χρόνος: περίπου 45–90 λεπτά (13 εκπαιδεύσεις). Μπορείς να το αφήσεις να τρέχει.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42
POWER = 0.25                       # βάρη κλάσεων (notebook 07)
EPOCHS, PATIENCE = 120, 10         # περισσότερες εποχές από πριν (ήταν 60 και 8)
CRITERION = "attack_cat macro-F1"  # με αυτό διαλέγουμε, στο validation

with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])
NUM_COLS, LOG_COLS = meta["num_cols"], meta["log_cols"]

FEATS_8 = ["dur", "dpkts", "tcprtt", "smean", "trans_depth", "ct_srv_src", "is_ftp_login", "is_sm_ips_ports"]

## 7.1 Δεδομένα (ίδια με τα notebooks 03 και 09) – μόνο τα 8 αριθμητικά
Οι στήλες `proto`, `service`, `state` **δεν** δίνονται στο δίκτυο.

In [ ]:
train_full = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
test_df = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")
for df in (train_full, test_df):
    df["attack_cat"] = df["attack_cat"].str.strip()
    df["severity"] = df["attack_cat"].map(meta["severity_map"])
if meta["drop_duplicates"]:
    train_full = train_full.drop_duplicates(subset=[c for c in train_full.columns if c != "id"])
train_df, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full["attack_cat"],
                                    random_state=meta["random_state"])
frames = {"train": train_df, "val": val_df, "test": test_df}

def encode_targets(df):
    return {"label": df["label"].to_numpy("int64"),
            "attack_cat": df["attack_cat"].map({c: i for i, c in enumerate(ATTACK_CLASSES)}).to_numpy("int64"),
            "severity": df["severity"].map({c: i for i, c in enumerate(SEVERITY_CLASSES)}).to_numpy("int64")}
y = {s: encode_targets(df) for s, df in frames.items()}

def targets(s):
    return [y[s]["label"].astype("float32").reshape(-1, 1), y[s]["attack_cat"].astype("int32"), y[s]["severity"].astype("int32")]

log_cols = [c for c in LOG_COLS if c in FEATS_8]
plain_cols = [c for c in FEATS_8 if c not in LOG_COLS]
steps = []
if log_cols:
    steps.append(("log", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                   ("scale", StandardScaler())]), log_cols))
if plain_cols:
    steps.append(("num", StandardScaler(), plain_cols))
preprocessor = ColumnTransformer(steps)
X = {"train": preprocessor.fit_transform(train_df).astype("float32")}      # fit ΜΟΝΟ στο train
X["val"], X["test"] = preprocessor.transform(val_df).astype("float32"), preprocessor.transform(test_df).astype("float32")
print("Σχήματα:", X["train"].shape, X["val"].shape, X["test"].shape)

## 7.2 Έλεγχος: ίδια 8 χαρακτηριστικά χωρίς τα διπλότυπα;
Στο notebook 08 οι συσχετίσεις υπολογίστηκαν σε όλο το train (με τις διπλότυπες γραμμές), ενώ η εκπαίδευση γίνεται χωρίς αυτές.
Ξανατρέχουμε την ίδια επιλογή (κατώφλι 0.5) στα δεδομένα χωρίς διπλότυπα.

In [ ]:
Xc = train_full[NUM_COLS].astype("float64").copy()
Xc[LOG_COLS] = np.log1p(Xc[LOG_COLS])
absc = Xc.corr().abs()

def select_features(threshold):
    kept = []
    for c in NUM_COLS:
        if all(absc.loc[c, k] <= threshold for k in kept):
            kept.append(c)
    return kept

sel = select_features(0.5)
v8 = absc.loc[FEATS_8, FEATS_8].values[np.triu_indices(len(FEATS_8), 1)]
print("Επιλογή χωρίς διπλότυπα:", sel)
print("Ίδια με τα 8 του notebook 08:", "ΝΑΙ" if sel == FEATS_8 else "ΟΧΙ")
if sel != FEATS_8:
    print("  μόνο στη νέα επιλογή:", [c for c in sel if c not in FEATS_8], "| μόνο στα 8:", [c for c in FEATS_8 if c not in sel])
print(f"Μέγιστο |r| ανάμεσα στα 8 (χωρίς διπλότυπα): {v8.max():.2f} | ζεύγη με |r| > 0.5: {int((v8 > 0.5).sum())}")

## 7.3 Στόχος: Random Forest στα ίδια 8 χαρακτηριστικά
Το υπολογίζουμε και στο validation, ώστε η σύγκριση με το δίκτυο να γίνεται εκεί που διαλέγουμε. (1–3 λεπτά.)

In [ ]:
def make_rf():
    return RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

rf_label = make_rf().fit(X["train"], y["train"]["label"])
rf_attack = make_rf().fit(X["train"], y["train"]["attack_cat"])
rf, rf_pred = {}, {}
for s in ("val", "test"):
    pl, pa = rf_label.predict_proba(X[s])[:, 1], rf_attack.predict(X[s])
    rf_pred[s] = pa
    rf[s] = {"label acc": accuracy_score(y[s]["label"], pl >= 0.5), "label ROC-AUC": roc_auc_score(y[s]["label"], pl),
             "attack_cat acc": accuracy_score(y[s]["attack_cat"], pa),
             "attack_cat macro-F1": f1_score(y[s]["attack_cat"], pa, average="macro")}
print(pd.DataFrame(rf).round(4).to_string())

## 7.4 Δίκτυο και κανόνας απόφασης (ίδια με πριν)
Το μόνο που αλλάζει ανά δοκιμή: το **μέγεθος** (`units`) και το **dropout**.

In [ ]:
def class_weights(y_cls, n_classes, power):
    counts = np.bincount(y_cls, minlength=n_classes).astype("float64")
    w = (len(y_cls) / (n_classes * np.maximum(counts, 1))) ** power
    return (w / np.average(w, weights=counts)).astype("float32")

def weighted_sparse_ce(class_w):
    w = keras.ops.convert_to_tensor(class_w, dtype="float32")
    def loss(y_true, y_pred):
        yy = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), "int32")
        return keras.losses.sparse_categorical_crossentropy(yy, y_pred) * keras.ops.take(w, yy)
    return loss

W_ATTACK = class_weights(y["train"]["attack_cat"], len(ATTACK_CLASSES), POWER)
W_SEV = class_weights(y["train"]["severity"], len(SEVERITY_CLASSES), POWER)

def build_model(units, dropout, lr=1e-3):
    inp = keras.Input(shape=(X["train"].shape[1],), name="flow_features")
    x = inp
    for i, u in enumerate(units):
        x = layers.Dense(u, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = layers.Dropout(dropout, name=f"dropout_{i+1}")(x)
    outs = [layers.Dense(1, activation="sigmoid", name="label")(x),
            layers.Dense(len(ATTACK_CLASSES), activation="softmax", name="attack_cat")(x),
            layers.Dense(len(SEVERITY_CLASSES), activation="softmax", name="severity")(x)]
    model = keras.Model(inp, outs, name="unsw_multitask_8num_tuned")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=["binary_crossentropy", weighted_sparse_ce(W_ATTACK), weighted_sparse_ce(W_SEV)],
                  loss_weights=[1.0, 1.0, 0.5])
    return model

def decide(p_label, p_attack, threshold=0.5):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)
    return is_attack.astype(int), attack, SEV_OF_ATTACK[attack]

def evaluate(model, split):
    p_label, p_attack, _ = model.predict(X[split], batch_size=1024, verbose=0)
    p_label = p_label.ravel()
    lab, att, sev = decide(p_label, p_attack)
    t = y[split]
    return {"label acc": accuracy_score(t["label"], lab),
            "label ROC-AUC": roc_auc_score(t["label"], p_label),
            "label recall": recall_score(t["label"], lab),
            "label FPR (ψευδείς συναγερμοί)": ((lab == 1) & (t["label"] == 0)).sum() / (t["label"] == 0).sum(),
            "attack_cat acc": accuracy_score(t["attack_cat"], att),
            "attack_cat macro-F1": f1_score(t["attack_cat"], att, average="macro"),
            "attack_cat macro-precision": precision_score(t["attack_cat"], att, average="macro", zero_division=0),
            "attack_cat macro-recall": recall_score(t["attack_cat"], att, average="macro"),
            "severity acc": accuracy_score(t["severity"], sev),
            "severity macro-F1": f1_score(t["severity"], sev, average="macro")}, att

def train(units, dropout, seed):
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = build_model(units, dropout)
    history = model.fit(X["train"], targets("train"), validation_data=(X["val"], targets("val")),
                        epochs=EPOCHS, batch_size=256, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
                                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)])
    return model, len(history.history["loss"])

# Σημείο εκκίνησης: το δίκτυο του notebook 09 (128-64-32, dropout 0.3, 60 εποχές)
old_model = keras.models.load_model(MODELS + "unsw_multitask_8num.keras", compile=False)
old_val, _ = evaluate(old_model, "val")
old_test, old_pred = evaluate(old_model, "test")
print("Δίκτυο notebook 09 | val:", {k: round(float(old_val[k]), 4) for k in ("label acc", "attack_cat acc", CRITERION)})

## 7.5 Αναζήτηση: 3 μεγέθη × 3 τιμές dropout
- **Μέγεθος:** μικρότερο (64-32), ίδιο (128-64-32), μεγαλύτερο (256-128-64).
- **Dropout:** 0.3 (όπως πριν), 0.1, 0.0. Με μόνο 8 εισόδους, το να «σβήνουμε» το 30% των νευρώνων ίσως εμποδίζει τη μάθηση αντί να τη βοηθά.
- **Εποχές:** έως 120, με διακοπή αν το validation loss δεν βελτιωθεί για 10 εποχές.

Κάθε φορά που ένα δίκτυο είναι το καλύτερο μέχρι στιγμής στο validation, αποθηκεύεται.

In [ ]:
UNITS = {"64-32": (64, 32), "128-64-32": (128, 64, 32), "256-128-64": (256, 128, 64)}
DROPOUTS = [0.3, 0.1, 0.0]
TUNED_PATH = MODELS + "unsw_multitask_8num_tuned.keras"

grid_val, grid_test, grid_pred, grid_epochs = {}, {}, {}, {}
best_cfg, best_score = None, -1.0
for u_name, units in UNITS.items():
    for dr in DROPOUTS:
        cfg = (u_name, dr)
        model, grid_epochs[cfg] = train(units, dr, SEED)
        grid_val[cfg], _ = evaluate(model, "val")
        grid_test[cfg], grid_pred[cfg] = evaluate(model, "test")
        flag = ""
        if grid_val[cfg][CRITERION] > best_score:
            best_cfg, best_score = cfg, grid_val[cfg][CRITERION]
            model.save(TUNED_PATH)
            flag = "  <- καλύτερο μέχρι τώρα"
        print(f"{u_name:>11}, dropout {dr}: {grid_epochs[cfg]:3d} εποχές | val label acc = {grid_val[cfg]['label acc']:.4f}, "
              f"attack_cat acc = {grid_val[cfg]['attack_cat acc']:.4f}, macro-F1 = {grid_val[cfg][CRITERION]:.4f}{flag}")

print(f"\nΕπιλογή στο validation: μέγεθος {best_cfg[0]}, dropout {best_cfg[1]}")
joblib.dump(preprocessor, MODELS + "unsw_preprocessor_8num_tuned.joblib")
with open(MODELS + "unsw_8num_tuned_config.json", "w", encoding="utf-8") as f:
    json.dump({"features": FEATS_8, "units": list(UNITS[best_cfg[0]]), "dropout": best_cfg[1], "class_weight_power": POWER,
               "epochs_max": EPOCHS, "patience": PATIENCE, "seed": SEED, "criterion": "validation " + CRITERION}, f,
              ensure_ascii=False, indent=2)

In [ ]:
idx = pd.MultiIndex.from_tuples(grid_val.keys(), names=["μέγεθος", "dropout"])
grid_table = pd.concat({"validation": pd.DataFrame(grid_val.values(), index=idx),
                        "test": pd.DataFrame(grid_test.values(), index=idx)}, axis=1).round(4)
grid_table.insert(0, "εποχές", list(grid_epochs.values()))
grid_table.to_csv(MODELS + "unsw_tuning_8num.csv")
show = ["label acc", "label ROC-AUC", "attack_cat acc", "attack_cat macro-F1"]
print("VALIDATION"); print(grid_table["validation"][show].assign(εποχές=grid_table["εποχές"]).to_string()); print()
print("TEST"); print(grid_table["test"][show].to_string())

SEQ_CMAP = LinearSegmentedColormap.from_list("seq", ["#f1f0ec", "#2a78d6"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metric in zip(axes, [CRITERION, "label acc"]):
    piv = pd.DataFrame({dr: {u: grid_val[(u, dr)][metric] for u in UNITS} for dr in DROPOUTS})
    sns.heatmap(piv, cmap=SEQ_CMAP, annot=True, fmt=".4f", annot_kws={"size": 10}, linewidths=2, linecolor="white",
                cbar_kws={"label": "validation"}, ax=ax)
    r, c = list(UNITS).index(best_cfg[0]), DROPOUTS.index(best_cfg[1])
    ax.add_patch(Rectangle((c, r), 1, 1, fill=False, edgecolor="#1f1f1d", linewidth=2.5))
    ax.set_title(f"{metric} (validation)"); ax.set_xlabel("dropout"); ax.set_ylabel("μέγεθος δικτύου")
    plt.setp(ax.get_yticklabels(), rotation=0)
fig.suptitle("Ρύθμιση του δικτύου στα 8 χαρακτηριστικά – το πλαίσιο δείχνει την επιλογή", y=1.02)
plt.tight_layout(); plt.savefig(FIG + "27_unsw_tuning_grid.png", dpi=200, bbox_inches="tight"); plt.show()

## 7.6 Είναι πραγματική η διαφορά; Έλεγχος με άλλους σπόρους
Δύο εκπαιδεύσεις του **ίδιου** δικτύου με διαφορετικό σπόρο τυχαιότητας δεν δίνουν ακριβώς το ίδιο αποτέλεσμα.
Αν η διαφορά «επιλεγμένο – αρχικό» είναι μικρότερη από αυτή τη φυσική διακύμανση, δεν μπορούμε να πούμε ότι η ρύθμιση βοήθησε.
Εκπαιδεύουμε το αρχικό μέγεθος/dropout και το επιλεγμένο με 3 σπόρους το καθένα.

In [ ]:
SEEDS = [SEED, 7, 123]
BASE_CFG = ("128-64-32", 0.3)
seed_rows = []
for cfg_name, cfg in [("αρχικό (128-64-32, dropout 0.3)", BASE_CFG), (f"επιλεγμένο ({best_cfg[0]}, dropout {best_cfg[1]})", best_cfg)]:
    for seed in SEEDS:
        if seed == SEED:
            v, t = grid_val[cfg], grid_test[cfg]                 # έχει ήδη εκπαιδευτεί στο 7.5
        else:
            m, _ = train(UNITS[cfg[0]], cfg[1], seed)
            v, _ = evaluate(m, "val"); t, _ = evaluate(m, "test")
        seed_rows.append({"δίκτυο": cfg_name, "σπόρος": seed,
                          **{f"val {k}": v[k] for k in show}, **{f"test {k}": t[k] for k in show}})
    print(cfg_name, "έτοιμο")
seed_table = pd.DataFrame(seed_rows)
seed_table.to_csv(MODELS + "unsw_tuning_8num_seeds.csv", index=False)
stats = seed_table.drop(columns="σπόρος").groupby("δίκτυο", sort=False).agg(["mean", "std"]).round(4)
print(stats.T.to_string())
if best_cfg == BASE_CFG:
    print("\nΗ αναζήτηση επέλεξε το αρχικό μέγεθος και dropout: το κέρδος (αν υπάρχει) προέρχεται μόνο από τις περισσότερες εποχές.")

## 7.7 Πριν, μετά και Random Forest (test)
Το test το βλέπουμε μόνο τώρα, για την αναφορά. Η επιλογή έχει ήδη γίνει στο validation.

In [ ]:
new_val, new_test, new_pred = grid_val[best_cfg], grid_test[best_cfg], grid_pred[best_cfg]
final = pd.DataFrame({("validation", "ΝΔ πριν"): {k: old_val[k] for k in show}, ("validation", "ΝΔ μετά"): {k: new_val[k] for k in show},
                      ("validation", "Random Forest"): rf["val"],
                      ("test", "ΝΔ πριν"): {k: old_test[k] for k in show}, ("test", "ΝΔ μετά"): {k: new_test[k] for k in show},
                      ("test", "Random Forest"): rf["test"]}).round(4)
print(final.to_string())
final.to_csv(MODELS + "unsw_tuning_8num_final.csv")
print(f"\nΨευδείς συναγερμοί (FPR) στο test: πριν {old_test['label FPR (ψευδείς συναγερμοί)']:.4f} → μετά {new_test['label FPR (ψευδείς συναγερμοί)']:.4f}")

series = [("ΝΔ πριν τη ρύθμιση", old_test, "#8a8a86"), ("ΝΔ μετά τη ρύθμιση", new_test, "#2a78d6"), ("Random Forest", rf["test"], "#eb6834")]
fig, ax = plt.subplots(figsize=(10.5, 4.6))
xpos = np.arange(len(show)); width = 0.26
for i, (name, res, color) in enumerate(series):
    bars = ax.bar(xpos + (i - 1) * width, [res[m] for m in show], width * 0.92, color=color, label=name)
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2, color="#3a3a38")
ax.set_xticks(xpos); ax.set_xticklabels(show)
ax.set_ylim(0, 1.1); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Σύστημα Α με 8 χαρακτηριστικά: νευρωνικό πριν και μετά τη ρύθμιση, και Random Forest (test)")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False)
plt.tight_layout(); plt.savefig(FIG + "28_unsw_tuning_before_after.png", dpi=200, bbox_inches="tight"); plt.show()

labels = list(range(len(ATTACK_CLASSES)))
rec = lambda p: recall_score(y["test"]["attack_cat"], p, labels=labels, average=None, zero_division=0)
recall_table = pd.DataFrame({"πλήθος": np.bincount(y["test"]["attack_cat"], minlength=len(ATTACK_CLASSES)),
                             "ΝΔ πριν": rec(old_pred), "ΝΔ μετά": rec(new_pred), "Random Forest": rec(rf_pred["test"])}, index=ATTACK_CLASSES)
print("\nRecall ανά τύπο επίθεσης (test):"); print(recall_table.round(3).to_string())

## Τι κρατάμε
- Το επιλεγμένο δίκτυο είναι στο `models/unsw_multitask_8num_tuned.keras` (ρυθμίσεις στο `unsw_8num_tuned_config.json`).
- Αν το κέρδος στο 7.6 είναι μικρότερο από τη διακύμανση μεταξύ σπόρων, γράφουμε τίμια ότι η ρύθμιση δεν άλλαξε ουσιαστικά το αποτέλεσμα: το όριο το θέτουν τα 8 χαρακτηριστικά, όχι το δίκτυο.